# **Clinical Trial Enrichment: Run 1 (Indication Mapping)**
---
This notebook orchestrates the first stage of the clinical enrichment pipeline. It filters the global ClinicalTrials.gov universe, defines the target trial cohort, and assembles the primary linguistic context used for mapping trials to the IHME GBD taxonomy.

# **0. Master Control: Reset & Configuration**
Define the execution mode, sampling parameters, and reset artifacts upfront.

In [1]:
# [CONTROL] Set to True to enable deletion of existing input contexts
RESET_PRODUCED_FILES = True

# [CONFIG] Toggle between full run and test sample
IS_PRODUCTION = True  # Set to True for full run
SAMPLE_SIZE = 300
RANDOM_STATE = 2000

import os
files_to_delete = [
        '../data/llm_in_01.csv',
    ]

if RESET_PRODUCED_FILES:
    for f in files_to_delete:
        if os.path.exists(f):
            os.remove(f)
            print(f"> Deleted: {f}")
    print("> Reset complete.")
else:
    print("> Reset skipped.")

> Deleted: ../data/llm_in_01.csv
> Reset complete.


# **1. Environment Setup**
Initialize libraries and project-specific utilities.

In [2]:
import pandas as pd
import os
import csv
import re
import sys
import json
import time
import asyncio
import logging
from tqdm.asyncio import tqdm
from dotenv import load_dotenv
from google import genai
from google.genai import types

# [STEP 1] Load environment variables from .env (API Keys)
load_dotenv()

# [STEP 2] Link the local source code folder so we can use the 'day_zero_reconstructor'
sys.path.append('..')
from src.prep.text_cleaning import day_zero_reconstructor
from src.prep.data_loader_clinpred import ClinicalTrialLoader

# [STEP 3] Define the working directories for raw data and processed results
DATA_PATH = '../data/'
OUTPUT_PATH = '../data/processed'

# [STEP 4] Global Newline Helper: Bypasses JSON parsing issues in notebook strings
NL = chr(10)

# [STEP 5] Utility function for robust CSV loading with specific clinical formatting
def safe_load(filename, cols=None):
    full_path = os.path.join(DATA_PATH, filename)
    # Match Strategy A: PERFECT from data_loader_clinpred.py
    params_perfect = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": csv.QUOTE_MINIMAL, "low_memory": False, "on_bad_lines": "warn"
    }
    # Match Strategy B: ROBUST
    params_robust = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
    }
    try:
        return pd.read_csv(full_path, usecols=cols, **params_perfect)
    except:
        return pd.read_csv(full_path, usecols=cols, **params_robust)

print("> SUCCESS: Environment Ready.")


> SUCCESS: Environment Ready.


# **2. Universe Filtering & Single Source of Truth**
We utilize the `ClinicalTrialLoader` to identify industry-led, interventional Phase 2/3 drug trials. This ensures that the enrichment universe exactly matches the training universe defined in the production source code.

In [3]:
# [STEP 1] Initialize Loader and pull the filtered universe
loader = ClinicalTrialLoader(DATA_PATH)
df = loader.get_filtered_universe()
print(f"> Filtered Universe: {len(df)} trials. (Sorted: Most Recent First)")

>>> Building the Raw Trial Universe (AACT Source of Truth)...
    [Filter] Initial Load: 582278 trials from studies.txt.
    [Filter] Kept 129086 Industry-led trials.
    [Filter] Kept 128154 Non-Expanded Access trials.
    [Filter] Kept 109636 Interventional trials.
    [Filter] Kept 102507 trials with allowed statuses: COMPLETED, TERMINATED, WITHDRAWN, RECRUITING, ACTIVE_NOT_RECRUITING, NOT_YET_RECRUITING, ENROLLING_BY_INVITATION
    [Filter] Kept 49282 Phase 2/3 focus trials (Excluded: EARLY_PHASE1, PHASE1, PHASE4, NA).
    [Filter] Kept 35527 trials within dynamic range 2009-2026.
    [Filter] Kept 34234 Drug/Biologic/Genetic trials.
    [Sanitizer] Dropping 168 trials terminated due to COVID/Logistics.
    [Filter] Kept 34066 trials after COVID sanitization.
> Filtered Universe: 34066 trials. (Sorted: Most Recent First)


# **3. Trial Selection & Sampling**
Using the configuration set in Section 1, this block determines whether to process the entire filtered universe or a random subset for validation purposes.

In [4]:
# [STEP 1] TRIAL SELECTION: Production vs. Testing
if IS_PRODUCTION:
    trial_df = df.copy()
    print(f"> PRODUCTION MODE: Processing all {len(trial_df)} trials.")
else:
    trial_df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).copy()
    print(f"> TESTING MODE: Processing {SAMPLE_SIZE} random trials.")

target_ids = trial_df['nct_id'].tolist()

> PRODUCTION MODE: Processing all 34066 trials.


# **4. Evidence Harvesting & Dictionary Optimization**
To maximize mapping accuracy, we load primary scientific evidence from conditions, summaries, and eligibility criteria. These datasets are converted into high-speed lookup dictionaries for efficient performance during context assembly.

In [5]:
# [STEP 2] Load and optimize primary evidence sources
df_cond = safe_load('conditions.txt', cols=['nct_id', 'name'])
df_sum = safe_load('brief_summaries.txt', cols=['nct_id', 'description'])
df_elig = safe_load('eligibilities.txt', cols=['nct_id', 'criteria'])
df_int = safe_load('interventions.txt', cols=['nct_id', 'intervention_type', 'name', 'description'])

# [STEP 3] OPTIMIZATION: Convert massive tables into Fast-Access Dictionaries (O(1) lookups)
print(">>> Building Fast-Access Dictionaries...")
from collections import defaultdict

def create_lookup(df, col_name=None):
    # This pre-sorts data into a Map {NCT_ID -> [List of Records]}
    lookup = defaultdict(list)
    for _, row in df.iterrows():
        lookup[row['nct_id']].append(row if col_name is None else row[col_name])
    return lookup

# Pre-calculate all lookups so the main loop runs in seconds, not hours
cond_lookup = create_lookup(df_cond, 'name')
sum_lookup = {row['nct_id']: row['description'] for _, row in df_sum.iterrows()}
elig_lookup = {row['nct_id']: row['criteria'] for _, row in df_elig.iterrows()}
int_lookup = create_lookup(df_int)

print(f"> Evidence Harvested for {len(target_ids)} trials.")


>>> Building Fast-Access Dictionaries...
> Evidence Harvested for 34066 trials.


# **5. Context Assembly & Production Export**
The final step iterates through the selected trial list, retrieves metadata, and applies the Day Zero Sanitizer. The resulting context blocks are saved to the primary input file for the enrichment runner.

In [6]:
results = []
for nct_id in target_ids:
    # [STEP 1] Retrieve the specific metadata for this trial
    row = trial_df[trial_df['nct_id'] == nct_id].iloc[0]

    # [STEP 2] Identity & Sanitization: Remove future timestamps/results to prevent data leakage
    clean_title = day_zero_reconstructor(row['official_title'], "title")

    # [STEP 3] Disease Information: Use Raw Names (conditions.txt) for maximum scientific nuance
    conds = " | ".join(list(set(cond_lookup.get(nct_id, []))))

    # [STEP 4] Intervention Evidence: Extract drug names and mechanism of action descriptions
    int_blocks = []
    for drug in int_lookup.get(nct_id, []):
        d = day_zero_reconstructor(drug['description'], "intervention") if pd.notna(drug['description']) else "No description"
        int_blocks.append(f"NAME: {drug['name']} ({drug['intervention_type']})" + NL + f"DESC: {d}")
    ints_string = (NL + "---" + NL).join(int_blocks)

    # [STEP 5] Protocol Essence: Clean the brief summary for LLM context window efficiency
    raw_summary = sum_lookup.get(nct_id, "")
    clean_summary = day_zero_reconstructor(raw_summary, "summary")

    # [STEP 6] Eligibility Rules: Include FULL Inclusion + Exclusion criteria (matching BioBERT input)
    raw_criteria = elig_lookup.get(nct_id, "")
    clean_criteria = day_zero_reconstructor(raw_criteria, "criteria")

    # [STEP 7] Final Super-Context Assembly (v2 - High Accuracy Focus)
    # Primary: [OFFICIAL_TITLE], [TRIAL_CONDITIONS_RAW], [PROTOCOL_SUMMARY]
    # Secondary: [ELIGIBILITY_CRITERIA_FULL], [PHASE], [INTERVENTION_DETAILS]
    context_body = f"""[TRIAL_START]
[NCT_ID]: {nct_id}
[START_YEAR]: {row['start_date'].year}
[PHASE]: {row['phase']}
[OFFICIAL_TITLE]: {clean_title}

[TRIAL_CONDITIONS_RAW]:
{conds}

[INTERVENTION_DETAILS]:
{ints_string}

[PROTOCOL_SUMMARY]:
{clean_summary[:5000]}...

[ELIGIBILITY_CRITERIA_FULL]:
{clean_criteria[:10000]}...
[TRIAL_END]"""

    results.append({"nct_id": nct_id, "context": context_body})

# [STEP 8] Save the assembled context blocks to a physical file for the enrichment stage
pd.DataFrame(results).to_csv(os.path.join(DATA_PATH, 'llm_in_01.csv'), index=False)
print(f"> Success: Assembled v2 context for {len(results)} trials.")


> Success: Assembled v2 context for 34066 trials.


---
## **Next Step: Run Enrichment Orchestrator**
After generating the context CSV in this notebook, shift to your terminal and execute the following command to start the LLM processing stage:

```bash
python3 src/prep/llm_in_01_run.py
```